<a id="understanding-quickstart"></a>
# VideoDB Understanding Quickstart

Create reusable timestamped outputs from speech and video frames, then inspect them.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="install"></a>
## 1. Install and connect

In [ ]:
!pip install -q videodb python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

<a id="video"></a>
## 2. Choose a video

The sample contains dialogue and clear scene changes, making it useful for speech and VLM analysis.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

<a id="run"></a>
## 3. Create an Understanding run

The transcript runs first. The VLM then uses sampled frames plus the aligned transcript for each scene.

> Go deeper: [segmentation and sampling](segmentation-and-sampling.ipynb) · [multi-analyzer pipelines](multi-analyzer-pipelines.ipynb)

In [ ]:
understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {"language": "en"},
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript"],
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {
                "model": "ultra",
                "prompt": "Describe what happens in this scene. Use the transcript only when it helps.",
                "schema": {"scene_description": "text"},
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding:", understanding.id)
print("Status:", understanding.status)

<a id="wait"></a>
## 4. Wait for completion

An Understanding can contain parallel and dependent analyzers. Wait for the run, then inspect each analyzer.

In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Final status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)

<a id="outputs"></a>
## 5. Fetch and display outputs

Analyzer output is returned as timestamped scenes. Each scene has `start`, `end`, and analyzer-specific `data`.

> For status handling, existing runs, and failures, see [Outputs and operations](outputs-and-operations.ipynb).

In [ ]:
import pandas as pd


def scene_rows(output):
    return [
        {
            "start": scene.get("start"),
            "end": scene.get("end"),
            **(scene.get("data") or {}),
        }
        for scene in (output or {}).get("scenes", [])
    ]


transcript_output = understanding.get_analyzer("transcript").get_output()
scene_output = understanding.get_analyzer("scene").get_output()

print("Transcript")
display(pd.DataFrame(scene_rows(transcript_output)).head())
print("Scene understanding")
display(pd.DataFrame(scene_rows(scene_output)).head())

<a id="existing"></a>
## 6. Resume an existing run

In [ ]:
same_understanding = video.get_understanding(understanding.id)
print(same_understanding)

for item in video.list_understandings():
    print(item.id, item.status)

<a id="next"></a>
## Next steps

| Goal | Notebook |
|---|---|
| Explore every analyzer | [Understanding guide map](README.md) |
| Tune scenes and frames | [Segmentation and sampling](segmentation-and-sampling.ipynb) |
| Compose dependent analyzers | [Multi-analyzer pipelines](multi-analyzer-pipelines.ipynb) |
| Use managed VLM models and schemas | [VLM with Managed Models](vlm/managed-models.ipynb) |
| Run a self-hosted VLM | [VLM with Sandbox Models](vlm/sandbox-models.ipynb) |
| Index these outputs | [Indexing guide](../indexing/indexing_guide.ipynb) |

<a id="cleanup"></a>
## Optional cleanup

In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete")